# Агенты

- [Архитектура надёжных AI-агентов — Сева Викулин](https://vikulin.ai/posts/ai-agent-architecture/)
- [The 8 Levels of Agentic Engineering — Bassim Eledath](https://www.bassimeledath.com/blog/levels-of-agentic-engineering)
- [Building effective agents — Anthropic](https://www.anthropic.com/engineering/building-effective-agents)
- [ReAct: Synergizing Reasoning and Acting in Language Models](https://huggingface.co/papers/2210.03629)
- [Hugging Face 🤗 AI Agents Course](https://huggingface.co/learn/agents-course/unit0/introduction)
- [Hugging Face 🤗 Building good agents](https://huggingface.co/docs/smolagents/tutorials/building_good_agents)
- [Introduction to LangGraph от LangChain Academy](https://academy.langchain.com/courses/intro-to-langgraph)

## Что такое агент

[Building effective agents от Anthropic](https://www.anthropic.com/engineering/building-effective-agents)

LLM pipeline / LLM workflow / LLM based сервис - с это сервис, в рамках которого LLM и прочие инструменты координируются по заранее определенным и зафиксированным путям (детерминированный граф), например, one-prompt application, RAG-сервис и прочее.

Agent system - это сервис, в рамках которого LLM-оркестратор самостоятельно принимает решения о том, какие шаги пайплайна выполнить, вопросы задать пользователю, в какие системы ходить, как разбить задачу на подзадачи и какие LLM-акторы ее должны выполнить (динамический граф). Агенты нужны для сложных задач (open-ended problems), например, понимание сложных входов, участие в рассуждениях и планировании, отладка ошибок. В идеале агентская система может полностью эмулировать специалиста в некоторой области. Агенты начинают свою работу либо с команды, либо интерактивной дискуссии с человеком.

| **LLM‑пайплайн** | **LLM‑агент** |
|------------------|---------------|
| Фиксированный граф | Динамический граф, строящийся в рантайме |
| Все шаги известны заранее | Шаги выбираются LLM по ходу работы |
| Нет циклов и саморефлексии | Может повторять шаги, откатываться, корректировать план |
| Ограниченная автономия | Высокая автономия; может взаимодействовать со средой |

Признаками агента являются:
* (Role) Умеет принимать на себя роль и выполнять задачу в ее рамках
* (Reasoning, Planning, Reflection) Понимает запрос, планирует, как выполнить задачу
* (Memory) Имеет знания о контексте и истории
* (Domain knowledge) Имеет знания о предметной области
* (Autonomy, Action, Ecosystem) Самостоятельно выполняет действия, взаимодействуя со средой

Фреймворки для создания агентных систем:
* [LangGraph (LangChain)](https://langchain-ai.github.io/langgraph/)
* [Smalagents by HuggigFace](https://huggingface.co/blog/smolagents)
* [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/)
* [AutoGen - Microsoft Research](https://www.microsoft.com/en-us/research/project/autogen/)
* [CrewAI](https://www.crewai.com/)
* [Haystack by Deepset](https://haystack.deepset.ai/)
* [Агенты Amazon Bedrock](https://aws.amazon.com/ru/bedrock/agents/)
* [Rivet (a visual programming environment for building AI agents with LLMs)](https://rivet.ironcladapp.com/)
* [Vellum (a visual programming environment for building AI agents with LLMs)](https://www.vellum.ai/)

## Анатомия AI-агента

Надёжный агент состоит из **5 функциональных блоков** и **2 защитных слоёв**.

### Функциональные блоки
* **LLM**
* **Оркестратор** - runtime-движок, который управляет бесконечным циклом работы агента: запускает компоненты с нужными аргументами, обрабатывает их выходные данные и перехватывает ошибки.
* **Контекстное окно** - память агента
    * Главный принцип: в момент запуска LLM в её памяти должна быть ровно та информация, которая необходима для текущего шага. 
    * Мусор в контексте — прямой путь к ненадёжному агенту. Чем длиннее контекст, тем хуже работает модель.
    * Алгоритмы управления памятью: сжатие, суммаризация, вытеснение старых диалогов.
* **Внешняя информация** - например, документы из RAG
* **Инструменты** - любые внешние программы, которые агент может вызывать для решения задачи: от калькулятора до API банковской системы. 
    * Инструментом может быть и другой агент — так появляются мультиагентные системы.
    * Для унификации подключений используется протокол MCP.

### Платформенные слои

Всё взаимодействие функциональных блоков проходит через оркестратор.                  
Чтобы агент был безопасным и управляемым, оркестратор помещается внутрь двух защитных слоёв:

1. **Безопасность**
    * Каждое действие проходит **премодерацию** до исполнения:
    - **Доступы** — имеет ли агент право читать данный файл?
    - **Prompt Injection** — нет ли попытки взломать инструкцию?
    - **Защита данных** — не утекают ли персональные данные? (их нужно шифровать или маскировать)      


2. **Observability (прозрачность)**
    * Необходимо логировать **всё**: каждый запуск LLM, каждый вызов инструмента, изменение состояния памяти. Без этого слоя система — чёрный ящик, с ним — есть возможность быстро находить причины ошибок и улучшать качество.

## Паттерны/парадигмы управления агентами (aka оркестрация)

От выбора оркестратора зависит, насколько хорошо будет работать система.

### [ReAct (Reason + Act)](https://huggingface.co/papers/2210.03629)
* **Базовая идея:** чередование шагов рассуждения и действия. Агент думает, затем выполняет действие, наблюдает за результатом и на основе этого переходит к следующему шагу. Цикл повторяется до достижения цели.
* **Когда использовать:** простые короткие задачи с небольшим числом инструментов.
* **Плюсы:** простота, прозрачность, адаптивность.
* **Минусы:** много вызовов LLM, риск зацикливания.

### Reflection & Reflexion
* **Базовая идея:** агент генерирует черновик ответа, затем критикует свою работу, находит ошибки и на основе этой критики пересматривает и улучшает ответ.
* **Reflection:** генерация → критика → пересмотр → повтор.
* **Reflexion:** более продвинутая версия, которая сохраняет критику как «вербальный след» для следующей итерации, позволяя агенту учиться на своих ошибках в рамках одной сессии.
* **Когда использовать:** задачи, где критично качество финального ответа: написание кода, статей, ответы на сложные вопросы.
* **Плюсы:** дешевый способ повысить качество, разделение ролей исполнителя и критика.
* **Минусы:** большое потребление токенов, качество критики зависит от возможностей LLM.

### Plan-and-Execute
* **Базовая идея:** сначала составляется детальный план всех шагов, затем план последовательно выполняется. Процесс разделён на две четкие фазы: планирование и исполнение.
* **Когда использовать:** многошаговые задачи с предсказуемой структурой, например, разработка архитектуры или написание кода.
* **Плюсы:** легче отслеживать прогресс, подходит для длинных траекторий.
* **Минусы:** план может устареть, затраты на этапе планирования.

## LLM Gateway (шлюз)

Нельзя пускать компоненты агента напрямую к моделям. Все запросы должны проходить через единый прокси-сервис (Gateway). Что должно быть «под капотом» у шлюза:

1. **Аналитика** — логируем, какую модель вызвали, сколько токенов съели и как долго отвечала
2. **Безопасность** — шифрование персональных данных, аутентификация, проверка на prompt injection
3. **Кеширование** — сохраняем популярные ответы
4. **Контролируемая деградация** — при отказе GPU переводим запрос на модель поменьше или в облако

## Контекстное окно: управление памятью

Контекстное окно — это оперативная память агента. Внутри: системный промпт, история работы, описание инструментов.

**Главный секрет надёжного агента:** перед каждым новым действием в контекстном окне должна быть ровно та информация, которая нужна именно сейчас.

### Три метода управления контекстом

1. **Хранение данных во внешних файлах** — объёмные данные выносятся во внешний файл, в контексте остаётся только идентификатор
2. **Сжатие контекста** — часть информации сжимается без потери смысла: либо правилами (результат инструмента десятишаговой давности уже не нужен), либо с помощью LLM (модель оценивает, что не пригодится, и суммаризует)
3. **Изоляция контекста (мультиагентность)** — если задача делится на независимые части, каждая решается в отдельном изолированном контексте

Эти методы хорошо комбинируются: хороший агент использует все три одновременно.

## Уровни агентной инженерии

Разрыв между возможностями моделей и нашей способностью их применять закрывается поэтапно. Bassim Eledath выделяет **8 уровней**, каждый из которых даёт скачок в производительности.

### Уровень 1–2: Tab Complete и Agent IDE
Началось с Copilot. Нажатие Tab дописывает код.  
Современные AI‑IDE (Cursor, Cline) связали чат с кодовой базой, сделав многофайловые правки простыми.  
Потолок – **контекст**: модель помогает только с тем, что видит. На этом уровне появляется **plan mode** – перевод идеи в пошаговый план для LLM.

### Уровень 3: Context Engineering
Контекстная инженерия – управление тем, что видит модель:
- System prompt и rules‑файлы (`.cursorrules`, `CLAUDE.md`)
- Описания инструментов – модель читает их, чтобы решать, какие вызывать
- Управление историей – чтобы агент не терял нить через 10 шагов
- Ограничение набора доступных инструментов за раз – слишком много вариантов перегружают модель

Сегодня модели стали терпимее к шумному контексту, но инженерия никуда не делась – фокус сместился с фильтрации плохого контекста на то, чтобы **нужный контекст был доступен в нужное время**.

### Уровень 4: Compounding Engineering
Цикл: **Plan → Delegate → Assess → Codify**.  
Ключевой шаг – **Codify**: записать, что сработало, что сломалось, какой паттерн использовать в следующий раз. Это делает процесс накопительным.  
Самый простой способ – обновлять `CLAUDE.md` или эквивалентный rules‑файл.  
**Важно**: не кодифицировать всё подряд – избыток инструкций даёт обратный эффект. Лучше создать среду, где LLM может легко обнаруживать полезный контекст (например, поддерживать актуальную папку `docs/`).  
Когда LLM ошибается, практикующие думают о недостающем контексте, а не о компетенции модели.

### Уровень 5: MCP и Skills
Уровни 3–4 решают проблему контекста, уровень 5 – проблему **возможностей**.  
**MCP** (Model Context Protocol) и кастомные навыки дают LLM доступ к:
- Базам данных и API
- CI/CD пайплайнам
- Дизайн‑системам
- Playwright для браузерного тестирования
- Slack для уведомлений

Когда несколько человек пишут свои версии одного навыка, стоит объединить их в общий реестр (например, внутренний маркетплейс).  
**Тренд**: LLM всё чаще используют CLI‑инструменты вместо MCP – они инжектят только релевантный вывод, а не полные схемы на каждом шаге, что экономит токены.

### Уровень 6: Harness Engineering и Automated Feedback Loops
Context engineering – о том, что видит модель. **Harness engineering** – о построении всей среды, инструментов и циклов обратной связи, позволяющих агентам работать надёжно **без вашего вмешательства**.  
OpenAI встроили Chrome DevTools, observability и браузерную навигацию в рантайм – по одному промпту агент может воспроизвести баг, записать видео, исправить, провалидировать через UI, открыть PR.  

Ключевая концепция – **backpressure**: автоматические механизмы обратной связи (типизация, тесты, линтеры, pre‑commit hooks), позволяющие агентам обнаруживать и исправлять ошибки без участия человека.  

Два принципа:
1. Проектируйте для **пропускной способности**, а не для совершенства – когда требуете совершенства, агенты топчутся на одном баге.
2. **Ограничения лучше инструкций** – пошаговые промпты устаревают; границы работают лучше.

### Уровень 7: Background Agents

Boris Cherny (создатель Claude Code) до сих пор начинает 80% задач в plan mode, но с каждым новым поколением моделей процент успешных one‑shot решений растёт.  
Мы приближаемся к точке, где plan mode как отдельный шаг с участием человека исчезает – **если** вы сделали работу на уровнях 3–6 (контекст чист, ограничения явны, инструменты описаны, циклы затянуты).  

Ключевой сдвиг: от «множества вкладок, которые я переключаю» к «работе, которая происходит без меня».  
Инструменты вроде **Dispatch** превращают сессию в командный центр: вы остаётесь в чистом окне, а воркеры делают тяжёлую работу в изолированных контекстах.  
**Ramp's Inspect** – альтернатива для долгосрочных автономных задач: каждая сессия агента запускается в облачной песочнице с полной средой разработки.  

**Важный паттерн**: используйте разные модели для разных задач (Opus – для реализации, Gemini – для исследования, Codex – для ревью). **Разделяйте исполнителя и ревьюера** – одна и та же модель предвзята.

### Уровень 8: Autonomous Agent Teams
На уровне 7 оркестратор распределяет работу по схеме «хаб‑спица». Уровень 8 убирает узкое место – агенты координируются **напрямую**: берут задачи, делятся находками, разрешают конфликты без единого оркестратора.  
Примеры: Anthropic использовала 16 параллельных агентов для сборки C‑компилятора; Cursor запускал сотни агентов на недели.  
Но проблемы остаются: без иерархии агенты становятся избегающими риска, ломают существующую функциональность.  

Для повседневной работы **уровень 7 – та точка, где реальная эффективность**.

## Model Context Protocol (MCP)

* [Introducing the Model Context Protocol](https://www.anthropic.com/news/model-context-protocol)
* [MCP для новичков](https://habr.com/ru/companies/raft/articles/927376/)

**MCP** — это открытый стандарт, разработанный Anthropic, который устанавливает универсальный способ подключения AI-ассистентов к системам, где хранятся данные: репозиториям, бизнес-инструментам, средам разработки и другим источникам.

По своей сути MCP — это API, разработанный специально для больших языковых моделей. Традиционные API созданы для языков программирования, а не для естественного языка: они требуют сложных промптов для обучения LLM схемам, подверженного ошибкам парсинга JSON, обработки аутентификации и управления состоянием между множественными вызовами. MCP решает эти проблемы, используя упрощенные, дружественные для LLM интерфейсы, описанные на естественном языке.

Вместо API-специфичного синтаксиса MCP предлагает последовательную структуру команд:

`[ДЕЙСТВИЕ] [ИМЯ_ИНСТРУМЕНТА] [ПАРАМЕТРЫ]`

Пример вызова инструмента:
`BOOK_FLIGHT to=Paris date=2023-12-01`

Инструменты определяются в стандартизированном формате манифеста:

In [1]:
# '''yaml
# tools:
#   - name: search_flights
#     description: "Найти доступные рейсы"
#     params:
#       - name: destination
#         type: string
#       - name: date
#         type: date
#   - name: book_flight
#     description: "Забронировать конкретный рейс"
#     params:
#       - name: flight_id
#         type: string
# '''

Ключевое отличие MCP от традиционных API — это управление состоянием, что отражено в термине «контекст» в названии протокола. В отличие от традиционных API (которые «забывают» вас быстрее золотой рыбки), MCP помнит контекст между вызовами.

### Архитектура MCP

- **MCP серверы** — раскрывают данные через стандартизированный протокол
- **MCP клиенты** — AI-приложения, которые подключаются к этим серверам

Разработчики могут как создавать свои MCP-серверы, так и использовать готовые.

### Готовые MCP-серверы

Anthropic предоставляет готовые MCP-серверы для популярных корпоративных систем:

- Google Drive
- Slack
- GitHub
- Git
- Postgres
- Puppeteer

Claude Desktop имеет встроенную поддержку MCP и поставляется с несколькими предустановленными MCP: Filesystem (чтение и запись локальных файлов), SQLite (запросы к базам данных), Web Search (поиск в интернете), Git (операции с репозиториями).

### Преимущества MCP

1. **Стандартизация** — единый протокол для всех интеграций LLM, аналогичный REST для API
2. **Управление состоянием** — помнит контекст между вызовами (пользователя, историю, репозиторий)
3. **Упрощенные команды** — естественный язык вместо сложных JSON-структур: `[ДЕЙСТВИЕ] [ИНСТРУМЕНТ] [ПАРАМЕТРЫ]`
4. **Двусторонняя связь** — безопасное соединение между данными и AI-инструментами
5. **Автоматическая аутентификация** — без управления токенами в каждом запросе
6. **Готовая экосистема** — серверы для Google Drive, Slack, GitHub, Postgres, Puppeteer
7. **Масштабируемость** — просто подключать новые источники данных

### Недостатки MCP

1. **Высокое потребление контекста** — до 72% окна еще до первого действия, 143K токенов для трех серверов
2. **Контекстное гниение** — падение точности выбора инструментов с 43% до <14% при росте числа инструментов
3. **Статическая загрузка** — все инструменты загружаются при старте, независимо от использования
4. **Пассивная архитектура** — не может инициировать действия, только отвечает на запросы
5. **Проблемы безопасности** — отравление инструментов, инъекция промптов, SSRF (36.7% серверов уязвимы), agentjacking (85% успешных атак)
6. **Нет идентичности** — отсутствие понятий пользователя, ролей и прав доступа
7. **Эксплуатационная сложность** — проблемы с подпроцессами, утечками памяти, нестабильной инициализацией
8. **Нет наблюдаемости** — отсутствие аудита, мониторинга и объяснимости действий
9. **Корпоративные проблемы** — не разработан для удаленных мультитенантных сред, создает потенциальный "черный ход"

> **Вывод**: MCP отлично подходит для локальных сценариев разработки, но для production-сред требует тщательной оценки безопасности.

## Ссылки

### Примеры MVP агентов

* [Repo for LangChain agent course](https://github.com/langchain-ai/langchain-academy)
* [gigachain_telegram_bot](https://github.com/Rai220/gigachain_telegram_bot/tree/main)
* [PydanticAI - Flight booking](https://ai.pydantic.dev/examples/flight-booking/)

### Multitool agents

* [langgraph-bigtool](https://github.com/langchain-ai/langgraph-bigtool)
* [LangGraph Multi-Agent Swarm](https://github.com/langchain-ai/langgraph-swarm-py)